# Citi Bike: preparation and evaluation

**Contribution owner: Tomislav Novosel.** Implementation, experiments and initial explanations were prepared with AI assistance on 9 October 2026. Human review and substantive changes should be recorded when they happen; see [the assistance record](../docs/AI_USE.md).

These are development experiments, not final-test results. The training/tuning dates end in June; July-September is development validation. October-December has not been scored.

The searches were executed with the helper scripts and their actual reports are saved. By default this notebook reviews those reports quickly. Set `CITIBIKE_RETRAIN=1` or use the runner's `--train` option to execute the search again. The XGBoost search needs a CUDA GPU; viewing results and running the saved app do not.


We verify the audited input and create only features known before each forecast. This notebook is the graph-free preparation step.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/citibike_forecast.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
from citibike_forecast import REPORTS, MODELS, development_data, chronological_cv
RUN_TRAINING = os.environ.get('CITIBIKE_RETRAIN') == '1'
def report(name):
    return json.loads((REPORTS / (name + '_run.json')).read_text(encoding='utf8'))
print('Mode:', 'execute training' if RUN_TRAINING else 'review actual saved experiments')
from citibike_forecast import prepare
X, y, training, validation, digest = prepare()
display(pd.DataFrame({'period': ['training', 'validation', 'reserved test'], 'start': ['2023-01-29','2023-07-01','2023-10-01'], 'end': ['2023-06-30','2023-09-30','2023-12-31'], 'days': [153,92,92]}))
display(X.head())
assert X.index.max() < pd.Timestamp('2023-10-01')


Mode: review actual saved experiments
Prepared 153 training and 92 validation dates; 92 final-test dates reserved.


,period,start,end,days
0,training,2023-01-29,2023-06-30,153
1,validation,2023-07-01,2023-09-30,92
2,reserved test,2023-10-01,2023-12-31,92


,weekday_sin,weekday_cos,year_sin,year_cos,weekend,lag_1,lag_2,lag_3,lag_7,lag_14,lag_21,lag_28,mean_3,mean_7,mean_14,mean_28,std_7,weekly_change,recent_change,holiday,before_holiday,after_holiday
date,,,,,,,,,,,,,,,,,,,,,,
2023-01-29,-0.781831,0.623490,0.463550,0.886071,1,66819.0,65614.0,64439.0,39438.0,41091.0,49454.0,50642.0,65624.000000,54891.714286,55276.428571,57480.500000,13043.477644,-1653.0,27381.0,0,0,0
2023-01-30,0.000000,1.000000,0.478734,0.877960,0,58041.0,66819.0,65614.0,38417.0,49763.0,61588.0,58241.0,63491.333333,57549.285714,56487.142857,57744.750000,11417.956268,-11346.0,19624.0,0,0,0
2023-01-31,0.781831,0.623490,0.493776,0.869589,0,70436.0,58041.0,66819.0,67647.0,64872.0,66674.0,51789.0,65098.666667,62123.428571,57963.785714,58180.285714,8993.264509,2775.0,2789.0,0,0,0
2023-02-01,0.974928,-0.222521,0.508671,0.860961,0,57398.0,70436.0,58041.0,41868.0,75820.0,67100.0,74454.0,61958.333333,60659.285714,57429.928571,58380.607143,8807.184400,-33952.0,15530.0,0,0,0
2023-02-02,0.433884,-0.900969,0.523416,0.852078,0,60429.0,57398.0,70436.0,64439.0,32358.0,51491.0,71440.0,62754.333333,63310.857143,56330.571429,57879.714286,4483.147409,32081.0,-4010.0,0,0,0


Lag 1 is yesterday, lag 7 is last week. Rolling means use shifted counts and therefore exclude the day being predicted. Calendar features are known in advance. The first 28 days supply history rather than targets.

In [2]:
display(pd.DataFrame({'feature': X.columns, 'missing': X.isna().sum().to_numpy()}))
contract = json.loads((REPORTS / 'contract.json').read_text())
display(pd.DataFrame(contract['folds']))
print('No random split: every training fold ends before its validation fold starts.')
print('The small tracked development snapshot allows a clean clone to train without downloading 2 GB again.')


No random split: every training fold ends before its validation fold starts.
The small tracked development snapshot allows a clean clone to train without downloading 2 GB again.


,feature,missing
0,weekday_sin,0
1,weekday_cos,0
2,year_sin,0
3,year_cos,0
4,weekend,0
5,lag_1,0
6,lag_2,0
7,lag_3,0
8,lag_7,0
9,lag_14,0


,training_start,training_end,validation_start,validation_end,training_rows,validation_rows
0,2023-01-29,2023-04-28,2023-04-29,2023-05-19,90,21
1,2023-01-29,2023-05-19,2023-05-20,2023-06-09,111,21
2,2023-01-29,2023-06-09,2023-06-10,2023-06-30,132,21
